# Stage 7: Rat GeneCompass Initialization and Fine-Tuning

> **This notebook shows the v2 build** (retrain v2, the working model since 2026-10-24; `config/pipeline_config.yaml` holds its paths). April's version of this notebook is in git history (commit `ad72cfa`); April's inputs are selected with `PIPELINE_PROFILE=config/profiles/april.yaml`.

## Stage Objective

Extend the pretrained GeneCompass model to rat genes and optimize it on a mixed-species corpus while retaining transferable human-mouse representations.

## Workflow

| Step | Operation | Analytical purpose |
|------|-----------|--------------------|
| 1 | Checkpoint extension | Expand the vocabulary and initialize rat token and species representations according to ortholog tier. |
| 2 | Frozen-encoder training | Train new embeddings and prior-knowledge projections while holding Transformer layers fixed. |
| 3 | Full-model training | Unfreeze the encoder and continue optimization at a lower learning rate. |
| 4 | Training audit | Evaluate loss trajectories, species composition, checkpoints, and final model files. |

## Inputs and Outputs

| Component | Description |
|-----------|-------------|
| Inputs | Stage 3 tokens, the Stage 5 v2 corpus mixed with GeneCompass's human and mouse cells (`mixed_species_dataset_v2`), Stage 6 outputs (T4 family/GRN rows only), and the pretrained GeneCompass checkpoint. |
| Primary outputs | `data/models/rat_genecompass_init_v2` and the final model `data/models/rat_genecompass_finetuned_v2/models/rat_phase2_mixed_species/models`. |

## Result Overview

The vocabulary is extended with the rat tokens, each phase covers the whole mixed-species dataset once, and the pre-registered checks C1 and C2 test whether the model learned rat expression without losing what GeneCompass knew about human and mouse (`docs/RETRAIN_V2_RESULTS.md`). The vocabulary size, steps per phase and the C1/C2 verdicts are printed below.

The sections below present the computational evidence in execution order. Every figure is printed by a code cell from the recorded stage outputs or the config; the markdown describes methods and points to those cells rather than quoting numbers.

In [1]:
import os, json

import yaml
MODELS_BASE = '../data/models'
FT_CFG = yaml.safe_load(open('../finetune/genecompass/configs/rat_finetune_v2.yaml'))
INIT_PROV = json.load(open(f'{MODELS_BASE}/rat_genecompass_init_v2/provenance_rat_model_init.json'))
FT_OUT = f"{MODELS_BASE}/rat_genecompass_finetuned_v2/models"

def last_checkpoint(run):
    """The highest-step checkpoint folder of a phase run (its trainer_state.json covers the whole phase)."""
    cks = [d for d in os.listdir(f'{FT_OUT}/{run}') if d.startswith('checkpoint-')]
    return f"{FT_OUT}/{run}/{max(cks, key=lambda d: int(d.split('-')[1]))}"

P1_STATE = f"{last_checkpoint('rat_phase1_freeze_mixed')}/trainer_state.json"
P2_STATE = f"{last_checkpoint('rat_phase2_mixed_species')}/trainer_state.json"
_vocab = json.load(open(f'{MODELS_BASE}/rat_genecompass_init_v2/config.json'))['vocab_size']
outputs = {
    'rat_genecompass_init_v2/':                 f'Step 1 — extended checkpoint ({_vocab:,} vocab)',
    'rat_genecompass_finetuned_v2/models/rat_phase1_freeze_mixed/models/': 'Phase 1 — final model',
    'rat_genecompass_finetuned_v2/models/rat_phase2_mixed_species/models/': 'Phase 2 — final model',
}
print(f"{'Path':<65} {'Size':>8}  Notes")
print('-' * 90)
for rel, desc in outputs.items():
    full = f'{MODELS_BASE}/{rel}'
    if os.path.isdir(full):
        total_b = 0
        for dirpath, _, fnames in os.walk(full):
            for fname in fnames:
                try: total_b += os.path.getsize(os.path.join(dirpath, fname))
                except: pass
        print(f"✓ {rel:<63} {total_b/1e9:>7.2f}G  {desc}")
    else:
        print(f"✗ {rel:<63} {'N/A':>8}  {desc}")
print()
# Checkpoint inventory for Phase 2
ckpt_dir = f'{MODELS_BASE}/rat_genecompass_finetuned_v2/models/rat_phase2_mixed_species'
ckpts = sorted([d for d in os.listdir(ckpt_dir) if d.startswith('checkpoint-')])
print(f"Phase 2 checkpoints kept ({len(ckpts)}; save_total_limit {FT_CFG['logging']['save_total_limit']}):")
for ck in ckpts:
    ck_path = f'{ckpt_dir}/{ck}'
    size_b = os.path.getsize(f'{ck_path}/pytorch_model.bin')
    print(f"  {ck:<30} {size_b/1e9:.2f} GB")

Path                                                                  Size  Notes
------------------------------------------------------------------------------------------
✓ rat_genecompass_init_v2/                                           1.39G  Step 1 — extended checkpoint (55,275 vocab)
✓ rat_genecompass_finetuned_v2/models/rat_phase1_freeze_mixed/models/    1.22G  Phase 1 — final model
✓ rat_genecompass_finetuned_v2/models/rat_phase2_mixed_species/models/    1.22G  Phase 2 — final model

Phase 2 checkpoints kept (5; save_total_limit 5):
  checkpoint-560000              1.22 GB
  checkpoint-570000              1.22 GB
  checkpoint-580000              1.22 GB
  checkpoint-590000              1.22 GB
  checkpoint-591763              1.22 GB


## Step 1 — Checkpoint Extension (`rat_model_init.py`)

GeneCompass_Base was pre-trained on human and mouse cells with its own token vocabulary. Stage 7 Step 1
extends this checkpoint to every rat gene in the universe: T1/T2 genes reuse their ortholog's token, and
new tokens are added for the T3 genes (a many-to-one human or mouse ortholog) and the T4 genes (no
ortholog). v2 init: job 11910364, 2026-10-08 (`slurm/finetune/init_v2.slurm`, `rat_finetune_v2.yaml`).
The vocabulary arithmetic is printed by the summary cell; the settings below are printed from the config
and the init provenance by the next cell.

### Initialization strategy (`rat_finetune_v2.yaml` → `initialization`, and the init provenance)

| Component | Strategy | Rationale |
|-----------|----------|-----------|
| T1–T2 tokens | Pre-trained GeneCompass rows, unchanged | Same token as the human/mouse ortholog |
| T3 word embeddings | **Clone** of the ortholog's embedding + N(0, `t3_noise_std`) noise | Warm start; noise breaks symmetry between paralogs |
| T4 word embeddings | **Random** N(0, `t4_word_emb_std`) | No ortholog → must learn from scratch |
| Species embedding (rat) | **Mouse + noise** N(0, `species_noise_std`) | Rat closer to mouse phylogenetically |
| T3 prior rows (all four knowledge types) | Copy of the ortholog's pre-trained rows | Already in GeneCompass's coordinates (fixed 2026-10-05; April left T3a rows zero) |
| T4 promoter, co-expression | **Zero** (`t4_zero_priors`) | The Stage 6 vectors are not in GeneCompass's coordinates (pre-registered test, 2026-10-06) |
| T4 gene family, GRN | Stage 6 vector by rat ID | The non-zero counts are in notebook 6's summary |

The co-expression rescale factor (`coexp_rescale_factor`) applies only to prior rows filled from Stage 6;
with T4 co-expression at zero, v2 has none.

In [2]:
print("rat_finetune_v2.yaml → initialization:")
for k, v in FT_CFG['initialization'].items():
    print(f"  {k:<22}: {v}")
print(f"\ninit provenance ({INIT_PROV['created'][:16]}, git {INIT_PROV['git_commit'][:8]}) → params:")
for k, v in INIT_PROV['params'].items():
    print(f"  {k:<22}: {v}")

rat_finetune_v2.yaml → initialization:
  t4_word_emb_init      : random
  t4_word_emb_std       : 0.02
  t3_warm_start         : True
  t3_noise_std          : 0.001
  species_init          : mouse_plus_noise
  species_noise_std     : 0.02
  coexp_rescale_factor  : 3.9
  t3_prior_source       : ortholog
  t4_zero_priors        : ['promoter', 'coexp']
  seed                  : 0

init provenance (2026-10-08T18:06, git 9620e612) → params:
  t3_prior_source       : ortholog
  t4_zero_priors        : ['promoter', 'coexp']
  coexp_rescale_factor  : 3.9
  t4_init_strategy      : random
  t4_init_std           : 0.02
  species_noise_std     : 0.02
  n_t3_ortholog         : 2243
  n_t3_missing          : 0


In [3]:
import json
with open('../data/models/rat_genecompass_init_v2/config.json') as f:
    cfg = json.load(f)
print("rat_genecompass_init_v2/config.json")
print("-" * 40)
for k, v in cfg.items():
    print(f"  {k:<35}: {v}")


rat_genecompass_init_v2/config.json
----------------------------------------
  architectures                      : ['BertForMaskedLM']
  attention_probs_dropout_prob       : 0.02
  classifier_dropout                 : None
  concat_values_first                : False
  emb_warmup_steps                   : 10000
  hidden_act                         : gelu
  hidden_dropout_prob                : 0.02
  hidden_size                        : 768
  initializer_range                  : 0.02
  intermediate_size                  : 3072
  layer_norm_eps                     : 1e-12
  max_position_embeddings            : 2048
  model_type                         : bert
  num_attention_heads                : 12
  num_hidden_layers                  : 12
  pad_token_id                       : 0
  position_embedding_type            : absolute
  torch_dtype                        : float32
  transformers_version               : 4.30.0
  type_vocab_size                    : 2
  use_cache                

In [4]:
import json
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

config_path = '../data/models/rat_genecompass_init_v2/config.json'
with open(config_path) as f:
    model_cfg = json.load(f)
import pickle
with open('../vendor/GeneCompass/prior_knowledge/human_mouse_tokens.pickle', 'rb') as f:
    old_vocab = len(pickle.load(f))   # the GeneCompass base vocabulary
new_vocab = model_cfg['vocab_size']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Stage 7: Rat GeneCompass Initialization', fontsize=13, fontweight='bold')
axes[0].axis('off')
architecture = [f"Vocabulary: {new_vocab:,}", f"Layers: {model_cfg['num_hidden_layers']}", f"Hidden size: {model_cfg['hidden_size']}", f"Attention heads: {model_cfg['num_attention_heads']}", f"Context: {model_cfg['max_position_embeddings']}"]
for i, label in enumerate(architecture):
    y = 0.82 - i * 0.15
    box = FancyBboxPatch((0.18, y), 0.64, 0.09, boxstyle='round,pad=0.02', fc='#E3F2FD', ec='#1976D2')
    axes[0].add_patch(box); axes[0].text(0.5, y + 0.045, label, ha='center', va='center')
axes[0].set(xlim=(0, 1), ylim=(0, 1)); axes[0].set_title('BERT architecture')
bars = axes[1].bar(['Human/mouse base', 'Rat-extended'], [old_vocab, new_vocab], color=['#607D8B', '#D32F2F'])
axes[1].bar_label(bars, fmt='%,.0f'); axes[1].set_ylabel('Vocabulary size'); axes[1].set_title(f'Vocabulary extension (+{new_vocab-old_vocab:,})'); axes[1].spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()


## Phase 1 — Frozen-Encoder Fine-Tuning

(SLURM job, node and wall time: see the Reproducibility Notes.)

### Strategy

Freeze all Transformer layers; only the **token embedding matrix** and **prior-knowledge
embedding projections** are trained. This allows the new rat tokens (T3, T4) and the rat species
embedding to stabilize before the full encoder is exposed to gradient updates.

### Config and coverage

The phase 1 settings (`rat_finetune_v2.yaml` → `phase1`) and the check that steps × global batch covers the
whole mixed dataset once are printed by the next cell. April's phase 1 covered only a quarter of its mix
because of a sampler bug (fixed in b460571).

### Results

The cell after it prints the loss at the first and last step and its means over the first and last 1,000
logged steps (single steps are noisy); with the encoder frozen, the fall in `id_loss` comes from the new
embeddings and prior projections alone.

In [5]:
from datasets import load_from_disk
N_MIX = load_from_disk('../data/training/mixed_species_dataset_v2').num_rows
GPUS = FT_CFG['distributed']['nproc_per_node'] * FT_CFG['distributed']['nnodes']

def show_phase(p):
    c = FT_CFG[p]
    keys = ['freeze_encoder', 'max_learning_rate', 'train_batch_size_per_gpu', 'num_train_epochs', 'warmup_steps',
            'lr_scheduler_type', 'dataset_directory', 'species_ratios', 'init_checkpoint']
    print(f"rat_finetune_v2.yaml → {p}:")
    for k in keys:
        if k in c:
            print(f"  {k:<26}: {c[k]}")
    st = json.load(open(P1_STATE if p == 'phase1' else P2_STATE))
    gb = c['train_batch_size_per_gpu'] * GPUS
    print(f"  global batch              : {c['train_batch_size_per_gpu']} × {GPUS} GPUs = {gb}")
    print(f"  coverage                  : {st['global_step']:,} steps × {gb} = {st['global_step'] * gb:,} rows "
          f"vs {N_MIX:,} in the mix ({st['global_step'] * gb / N_MIX:.3f} passes)")

show_phase('phase1')

/depot/reese18/apps/motrpac-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rat_finetune_v2.yaml → phase1:
  freeze_encoder            : True
  max_learning_rate         : 0.0001
  train_batch_size_per_gpu  : 16
  num_train_epochs          : 1
  warmup_steps              : 1000
  lr_scheduler_type         : linear
  dataset_directory         : data/training/mixed_species_dataset_v2
  species_ratios            : {'rat': 0.4, 'human': 0.45, 'mouse': 0.15}


  global batch              : 16 × 4 GPUs = 64
  coverage                  : 517,793 steps × 64 = 33,138,752 rows vs 33,138,703 in the mix (1.000 passes)


In [6]:
import json
import numpy as np
with open(P1_STATE) as f:
    state = json.load(f)
log = state['log_history']
id_losses  = [e['id_loss']   for e in log if 'id_loss' in e]
val_losses = [e['value_loss'] for e in log if 'value_loss' in e]
print(f"Phase 1 global_step: {state['global_step']:,}")
print(f"Phase 1 epoch:       {state['epoch']:.6f}")
print(f"id_loss:   first step={id_losses[0]:.4f}   last step={id_losses[-1]:.4f}   min={min(id_losses):.4f}")
print(f"id_loss:   mean first 1,000={np.mean(id_losses[:1000]):.4f}   mean last 1,000={np.mean(id_losses[-1000:]):.4f}")
print(f"val_loss:  mean first 1,000={np.mean(val_losses[:1000]):.6f}  mean last 1,000={np.mean(val_losses[-1000:]):.6f}")
print(f"Steps with losses logged: {len(id_losses):,}  (log entries: {len(log):,})")

Phase 1 global_step: 517,793
Phase 1 epoch:       1.000000
id_loss:   first step=3.7408   last step=3.6564   min=2.8309
id_loss:   mean first 1,000=4.1201   mean last 1,000=3.3277
val_loss:  mean first 1,000=0.012211  mean last 1,000=0.000410
Steps with losses logged: 517,793  (log entries: 528,148)


In [7]:
import json
import matplotlib.pyplot as plt
import numpy as np

state_path = P1_STATE
with open(state_path) as f: phase_state = json.load(f)
entries = [e for e in phase_state['log_history'] if 'id_loss' in e and 'value_loss' in e]
steps = np.array([e['step'] for e in entries]); id_loss = np.array([e['id_loss'] for e in entries]); value_loss = np.array([e['value_loss'] for e in entries])
window = 1000
kernel = np.ones(window) / window
smooth_steps = steps[window-1:]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Phase 1 Training Curves ({window}-step moving average)', fontsize=13, fontweight='bold')
axes[0].plot(smooth_steps, np.convolve(id_loss, kernel, mode='valid'), color='#1976D2')
axes[1].plot(smooth_steps, np.convolve(value_loss, kernel, mode='valid'), color='#D32F2F')
axes[0].set(ylabel='ID loss', title='Masked gene identity loss'); axes[1].set(ylabel='Value loss', title='Expression value loss')
for ax in axes: ax.set_xlabel('Training step'); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## Phase 2 — Full Model Fine-Tuning

(SLURM job, nodes and wall time: see the Reproducibility Notes. The job was requeued once after a node
failure and resumed from a saved checkpoint; the final `trainer_state.json` logs every step once, and the
first attempt's logged steps before the resume point equal the final run's (C1a, printed below).)

### Strategy

All Transformer encoder layers are **unfrozen**. A lower learning rate than phase 1 prevents catastrophic
forgetting of the human/mouse pre-training while allowing the full model to specialize for rat biology.

Phase 2 initializes from **phase 1's final model** (`rat_phase1_freeze_mixed/models`).

### Config and results

The next cell prints the phase 2 settings and coverage; the cell after it prints the losses and the
phase 1 → phase 2 hand-off. The training loss improves only a little in phase 2; what it does on held-out
cells is answered by the C1/C2 probe further down.

> **Final model:** `data/models/rat_genecompass_finetuned_v2/models/rat_phase2_mixed_species/models/`
> (byte-identical to the last phase 2 checkpoint)

In [8]:
show_phase('phase2')

rat_finetune_v2.yaml → phase2:
  freeze_encoder            : False
  max_learning_rate         : 1e-05
  train_batch_size_per_gpu  : 14
  num_train_epochs          : 1
  warmup_steps              : 2000
  lr_scheduler_type         : linear
  dataset_directory         : data/training/mixed_species_dataset_v2
  species_ratios            : {'rat': 0.4, 'human': 0.45, 'mouse': 0.15}
  init_checkpoint           : data/models/rat_genecompass_finetuned_v2/models/rat_phase1_freeze_mixed/models


  global batch              : 14 × 4 GPUs = 56
  coverage                  : 591,763 steps × 56 = 33,138,728 rows vs 33,138,703 in the mix (1.000 passes)


In [9]:
import json
import numpy as np
with open(P2_STATE) as f:
    state = json.load(f)
with open(P1_STATE) as f:
    state1 = json.load(f)
log = state['log_history']
id_losses  = [e['id_loss']   for e in log if 'id_loss' in e]
val_losses = [e['value_loss'] for e in log if 'value_loss' in e]
id1 = [e['id_loss'] for e in state1['log_history'] if 'id_loss' in e]
print(f"Phase 2 global_step: {state['global_step']:,}")
print(f"Phase 2 epoch:       {state['epoch']:.6f}")
print(f"id_loss:   first step={id_losses[0]:.4f}   last step={id_losses[-1]:.4f}   min={min(id_losses):.4f}")
print(f"id_loss:   mean first 1,000={np.mean(id_losses[:1000]):.4f}   mean last 1,000={np.mean(id_losses[-1000:]):.4f}")
print(f"val_loss:  mean first 1,000={np.mean(val_losses[:1000]):.6f}  mean last 1,000={np.mean(val_losses[-1000:]):.6f}")
steps = [e['step'] for e in log if 'id_loss' in e]
print(f"Steps with losses logged: {len(steps):,}  (distinct: {len(set(steps)):,})")
print()
print("Phase 1 → Phase 2 id_loss handoff (1,000-step means):")
print(f"  Phase 1 last 1,000:   {np.mean(id1[-1000:]):.4f}")
print(f"  Phase 2 first 1,000:  {np.mean(id_losses[:1000]):.4f}")
print(f"  Phase 2 last 1,000:   {np.mean(id_losses[-1000:]):.4f}")

Phase 2 global_step: 591,763
Phase 2 epoch:       1.000000
id_loss:   first step=3.7001   last step=3.5272   min=2.7699
id_loss:   mean first 1,000=3.3327   mean last 1,000=3.2848
val_loss:  mean first 1,000=0.000421  mean last 1,000=0.000309
Steps with losses logged: 591,763  (distinct: 591,763)

Phase 1 → Phase 2 id_loss handoff (1,000-step means):
  Phase 1 last 1,000:   3.3277
  Phase 2 first 1,000:  3.3327
  Phase 2 last 1,000:   3.2848


In [10]:
import json
import matplotlib.pyplot as plt
import numpy as np

state_path = P2_STATE
with open(state_path) as f: phase_state = json.load(f)
entries = [e for e in phase_state['log_history'] if 'id_loss' in e and 'value_loss' in e]
steps = np.array([e['step'] for e in entries]); id_loss = np.array([e['id_loss'] for e in entries]); value_loss = np.array([e['value_loss'] for e in entries])
window = 1000
kernel = np.ones(window) / window
smooth_steps = steps[window-1:]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Phase 2 Training Curves ({window}-step moving average)', fontsize=13, fontweight='bold')
axes[0].plot(smooth_steps, np.convolve(id_loss, kernel, mode='valid'), color='#00897B')
axes[1].plot(smooth_steps, np.convolve(value_loss, kernel, mode='valid'), color='#F57C00')
axes[0].set(ylabel='ID loss', title='Masked gene identity loss'); axes[1].set(ylabel='Value loss', title='Expression value loss')
for ax in axes: ax.set_xlabel('Training step'); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## Mixed-species dataset, final loss summary and the C1/C2 check

The pie shows the rows actually in `mixed_species_dataset_v2` (from its build provenance; the rat and human
rows repeat their source cells to reach the configured species mix; the counts are printed below).

**Did the model learn rat without forgetting human and mouse?** Pre-registered C1/C2 (job 12011656,
`docs/RETRAIN_V2_RESULTS.md`): the same held-out cells per species and the same masked tokens for every
model, scored by `finetune/genecompass/score_prereg_c1c2.py` into `data/results/retrain_v2/mlm/`. The next
cell prints the table from the per-model JSONs and the verdict lines of `score_c1c2.txt`:

- **C1a:** every step of both phases is logged once with finite losses.
- **C1b:** on the rat probe, v2 beats v2 init on gene-ID loss and value loss.
- **C2:** v2 stays within +10% of base on human and mouse, for both losses.

In [11]:
MLM = '../data/results/retrain_v2/mlm'
models = [('base', 'base'), ('init_v2', 'v2 init'), ('v2_phase1', 'v2 after phase 1'), ('v2', 'v2')]
res = {m: json.load(open(f'{MLM}/{m}.json'))['species'] for m, _ in models}
print(f"{'Model':<18} {'Species':<8} {'Gene-ID loss':>12} {'Value loss':>11} {'Top-1':>7} {'ID vs base':>11} {'Value vs base':>14}")
for m, label in models:
    for sp, r in res[m].items():
        if m == 'base' and sp == 'rat':
            continue
        b = res['base'].get(sp) if sp != 'rat' else None
        vs = (f"{100 * (r['id_loss'] / b['id_loss'] - 1):+.2f}%", f"{100 * (r['value_loss'] / b['value_loss'] - 1):+.2f}%") if b and m != 'base' else ('', '')
        print(f"{label:<18} {sp:<8} {r['id_loss']:>12.4f} {r['value_loss']:>11.3e} {100 * r['top1_acc']:>6.2f}% {vs[0]:>11} {vs[1]:>14}")
print(f"\n(cells per species: {res['v2']['rat']['cells']:,})\n")
print("score_c1c2.txt:")
for line in open(f'{MLM}/score_c1c2.txt'):
    if line.startswith(('C1', 'C2')):
        print("  " + line.rstrip())

Model              Species  Gene-ID loss  Value loss   Top-1  ID vs base  Value vs base
base               human          3.4444   1.728e-04  21.25%                           
base               mouse          3.4215   1.953e-04  21.48%                           
v2 init            human          3.5323   1.728e-04  20.58%      +2.55%         +0.00%
v2 init            mouse          3.5177   1.953e-04  20.71%      +2.81%         +0.00%
v2 init            rat            7.1332   1.330e-01  13.36%                           
v2 after phase 1   human          3.4634   1.740e-04  21.10%      +0.55%         +0.67%
v2 after phase 1   mouse          3.4854   2.094e-04  20.97%      +1.87%         +7.24%
v2 after phase 1   rat            3.1565   5.421e-04  24.27%                           
v2                 human          3.4462   1.680e-04  21.22%      +0.05%         -2.81%
v2                 mouse          3.4667   2.002e-04  21.22%      +1.32%         +2.53%
v2                 rat          

In [12]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open('../data/training/mixed_species_dataset_v2/provenance_build_mixed_species_dataset.json') as f:
    mix = json.load(f)['params']
rows = {s: mix['counts'][s]['rows'] for s in ('human', 'mouse', 'rat')}
print(f"mixed_species_dataset_v2: {mix['n_rows']:,} rows (seed {mix['seed']}, {mix['stopping_strategy']})")
for s in ('human', 'mouse', 'rat'):
    c = mix['counts'][s]
    print(f"  {s:<6} {c['rows']:>11,} rows ({c['rows']/mix['n_rows']:.1%})  from {c['distinct_source_cells']:,} distinct cells")
base = '../data/models/rat_genecompass_finetuned_v2/models'
states = {}
for label, path in [('Phase 1', P1_STATE), ('Phase 2', P2_STATE)]:
    with open(path) as f: states[label] = json.load(f)
final_losses = []
for state in states.values():
    vals = [e['id_loss'] for e in state['log_history'] if 'id_loss' in e]
    final_losses.append(float(np.mean(vals[-1000:])))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Stage 7: Mixed-Species Training Summary', fontsize=13, fontweight='bold')
species = ['Human', 'Mouse', 'Rat']; values = [rows[s.lower()] for s in species]
axes[0].pie(values, labels=species, autopct='%1.0f%%', startangle=90, colors=['#1976D2', '#00897B', '#D32F2F'], wedgeprops={'width': 0.42, 'edgecolor': 'white'})
axes[0].set_title('Rows in the v2 mix')
bars = axes[1].bar(list(states), final_losses, color=['#7B1FA2', '#F57C00'])
axes[1].bar_label(bars, fmt='%.3f'); axes[1].set_ylabel('Mean ID loss, final 1,000 steps'); axes[1].set_title('End-of-phase identity loss (training)'); axes[1].spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

mixed_species_dataset_v2: 33,138,703 rows (seed 42, all_exhausted)
  human   14,913,398 rows (45.0%)  from 5,000,000 distinct cells
  mouse    4,969,697 rows (15.0%)  from 4,969,697 distinct cells
  rat     13,255,608 rows (40.0%)  from 6,803,206 distinct cells


In [13]:
import os
import json, pickle
import numpy as np
from datasets import load_from_disk
_M = '../data/models/rat_genecompass_finetuned_v2/models'
_cfg = json.load(open('../data/models/rat_genecompass_init_v2/config.json'))
_tiers = json.load(open('../data/training/ortholog_mappings/mapping_statistics.json'))['tier_distribution']
_base = len(pickle.load(open('../vendor/GeneCompass/prior_knowledge/human_mouse_tokens.pickle', 'rb')))
_n_t3 = _tiers['T3a_human_multi'] + _tiers['T3b_mouse_multi']
_n_t4 = _tiers['T4_new_token']
_rows = load_from_disk('../data/training/mixed_species_dataset_v2').num_rows
def _phase(name):
    ck = sorted((d for d in os.listdir(f'{_M}/{name}') if d.startswith('checkpoint-')), key=lambda d: int(d.split('-')[1]))[-1]
    st = json.load(open(f'{_M}/{name}/{ck}/trainer_state.json'))
    idl = [e['id_loss'] for e in st['log_history'] if 'id_loss' in e]
    return st['global_step'], np.mean(idl[:1000]), np.mean(idl[-1000:])
s1, a1, b1 = _phase('rat_phase1_freeze_mixed')
s2, a2, b2 = _phase('rat_phase2_mixed_species')

print("=" * 65)
print("STAGE 7 SUMMARY — GeneCompass Rat Fine-Tuning (v2)")
print("=" * 65)
print()
print("Model architecture:")
print(f"  Type:           BERT ({_cfg['architectures'][0] if _cfg.get('architectures') else 'BertForMaskedLM'})")
print(f"  Layers:         {_cfg['num_hidden_layers']}")
print(f"  Hidden size:    {_cfg['hidden_size']}")
print(f"  Attention heads: {_cfg['num_attention_heads']}")
print(f"  Context length: {_cfg['max_position_embeddings']} tokens")
print(f"  Vocab size:     {_cfg['vocab_size']:,}  (extended from {_base:,}: {_n_t3:,} T3 + {_n_t4:,} T4)")
print("  Prior knowledge: T1-T2 pre-trained rows; T3 ortholog rows; T4 family + GRN from Stage 6,")
print("                   promoter + co-expression zero")
print()
print(f"Training (mixed_species_dataset_v2, {_rows:,} rows, each phase one pass):")
print(f"  Phase 1  (freeze)  : {s1:,} steps | id_loss {a1:.2f}→{b1:.2f} (1,000-step means)")
print(f"  Phase 2  (full)    : {s2:,} steps | id_loss {a2:.2f}→{b2:.2f} (1,000-step means)")
print("  Wall time: from SLURM accounting, not recorded in the outputs (Reproducibility Notes)")
print()
_verdict = [l.strip() for l in open('../data/results/retrain_v2/mlm/score_c1c2.txt') if l.startswith('C1a')]
print(f"Pre-registered checks (score_c1c2.txt): {_verdict[-1] if _verdict else 'not found'}")
print()
print("Final model:")
print("  data/models/rat_genecompass_finetuned_v2/models/rat_phase2_mixed_species/models/")
print()
print("Downstream:")
print("  Stage 9  → embed MoTrPAC pseudo-cells (Aim 2)")
print("  Stage 12 → re-express as human and embed (Aim 3)")

STAGE 7 SUMMARY — GeneCompass Rat Fine-Tuning (v2)

Model architecture:
  Type:           BERT (BertForMaskedLM)
  Layers:         12
  Hidden size:    768
  Attention heads: 12
  Context length: 2048 tokens
  Vocab size:     55,275  (extended from 50,558: 2,243 T3 + 2,474 T4)
  Prior knowledge: T1-T2 pre-trained rows; T3 ortholog rows; T4 family + GRN from Stage 6,
                   promoter + co-expression zero

Training (mixed_species_dataset_v2, 33,138,703 rows, each phase one pass):
  Phase 1  (freeze)  : 517,793 steps | id_loss 4.12→3.33 (1,000-step means)
  Phase 2  (full)    : 591,763 steps | id_loss 3.33→3.28 (1,000-step means)
  Wall time: from SLURM accounting, not recorded in the outputs (Reproducibility Notes)

Pre-registered checks (score_c1c2.txt): C1a PASS | C1b PASS | C2 PASS

Final model:
  data/models/rat_genecompass_finetuned_v2/models/rat_phase2_mixed_species/models/

Downstream:
  Stage 9  → embed MoTrPAC pseudo-cells (Aim 2)
  Stage 12 → re-express as human and 

## Stage Summary

Stage 7 extends the pretrained vocabulary with the rat tokens, stabilizes new representations with a frozen
encoder, and then fine-tunes the full model, each phase covering the whole mix once. The training curves
are generated directly from checkpoint state files; whether the model learned rat without forgetting human
and mouse is answered by the held-out C1/C2 probe, whose verdicts are printed above.

---

## Reproducibility Notes

### Environment

| Item | Value |
|------|-------|
| Cluster | Purdue Gilbreth (SLURM) |
| GPU | 4 × A100-40GB (gilbreth-n012, n011, n007) |
| Python env | `motrpac-env` (venv) |
| Framework | PyTorch + HuggingFace Transformers (versions in each `config.json`, printed below) |
| Precision | fp16 (mixed precision; `training.fp16` in the config) |
| CPUs / memory per training job | 32 / 256 GB (SLURM accounting) |

### SLURM job history (from SLURM accounting; not recorded in the outputs)

| Phase | Job ID | Node | Start (EDT) | End (EDT) | Elapsed | Exit |
|-------|--------|------|-------------|-----------|---------|------|
| Init ✓ | 11910364 | gilbreth-n002 | 2026-10-08 18:06 | 18:06 | 30 s | 0 |
| Mixed dataset ✓ | 11911302 | gilbreth-n000 | 2026-10-08 23:56 | 10-09 01:07 | 1.2 h | 0 |
| Phase 1 ✓ | **11917121** | gilbreth-n012 | 2026-10-10 19:01 | 10-15 18:39 | **119.6 h** | 0 |
| Phase 2 (node failure) | 11961217 | gilbreth-n011 | 2026-10-16 19:09 | 10-21 21:10 | 122.0 h | NODE_FAIL |
| Phase 2 ✓ (requeued) | **11961217** | gilbreth-n007 | 2026-10-22 09:31 | 10-23 11:17 | **25.8 h** | 0 |

> April's model (jobs 10515821 + 10533641, `rat_genecompass_finetuned/`) is described in this notebook's
> April version (git history, `ad72cfa`).

### Configuration

The full configuration is `finetune/genecompass/configs/rat_finetune_v2.yaml`; the next cell prints its
`model`, `training`, `mixing` and `logging` sections (the `initialization`, `phase1` and `phase2` sections
are printed above).

### Determinism

**Non-deterministic.** Multi-GPU DDP training with fp16 introduces non-deterministic
floating-point accumulation order. Exact reproduction of the loss trajectory requires:
- Same GPU hardware (NVIDIA model + driver)
- Same `CUBLAS_WORKSPACE_CONFIG=:4096:8` (cuBLAS determinism mode)
- Single GPU (`--nproc_per_node=1`) — eliminates NCCL non-determinism

For publication reporting, the final `rat_phase2_mixed_species/models` (= the last phase 2 checkpoint) is
the authoritative artifact; its md5 is pinned in `config/deposit_md5.tsv`. The phase 2 requeue resumed from
a saved checkpoint, and the steps both attempts logged before it are equal (C1a), so the resume did not
change the trajectory.

### WandB run

Phase 1: `wandb.ai/reese18-purdue-university/motrpac-genecompass-rat/runs/vbjj3oq6`
Phase 2 (requeued run): `wandb.ai/reese18-purdue-university/motrpac-genecompass-rat/runs/5w3z3mzm`

### Re-running Stage 7

```bash
cd ..

# Step 1: extend checkpoint (seconds; writes a new directory, never overwrites)
sbatch slurm/finetune/init_v2.slurm

# Mixed-species dataset (rat v2 corpus + GeneCompass human/mouse)
python finetune/genecompass/build_mixed_species_dataset.py --rat-dataset data/training/tokenized_corpus_v2/dataset \
    --human-dataset vendor/GeneCompass/data/randsel_500w_human --mouse-dataset vendor/GeneCompass/data/randsel_500w_mouse \
    --output data/training/mixed_species_dataset_v2

# Phase 1, then phase 2 (4 × A100, about five days each; resumes from the latest checkpoint on resubmission)
CONFIG=finetune/genecompass/configs/rat_finetune_v2.yaml OUT=data/models/rat_genecompass_finetuned_v2 PHASE=1 sbatch slurm/finetune/stage7.slurm
CONFIG=finetune/genecompass/configs/rat_finetune_v2.yaml OUT=data/models/rat_genecompass_finetuned_v2 PHASE=2 sbatch slurm/finetune/stage7.slurm
```

### Mixed-species leakage safeguard

The MoTrPAC exercise transcriptomics data (Aim 2 / Aim 3 target) is **not** included
in the fine-tuning corpus. The model is trained on:
- the Stage 5 v2 rat corpus (files and studies in notebook 5; GSE242354 excluded)
- GeneCompass human/mouse random-selection datasets (randsel_500w_*)

MoTrPAC data is held out until Stage 8 (deconvolution) and Stage 9 (embedding).

In [14]:
import platform
print(f"Python {platform.python_version()}")
for label, path in [('v2 init', f'{MODELS_BASE}/rat_genecompass_init_v2'), ('final model', f'{FT_OUT}/rat_phase2_mixed_species/models')]:
    print(f"transformers in {label} config.json: {json.load(open(f'{path}/config.json')).get('transformers_version')}")
print()
for sec in ('model', 'training', 'mixing', 'logging', 'distributed'):
    print(f"{sec}: {FT_CFG[sec]}")

Python 3.12.11
transformers in v2 init config.json: 4.30.0
transformers in final model config.json: 4.57.6

model: {'base_checkpoint': 'vendor/GeneCompass/pretrained_models/GeneCompass_Base', 'init_checkpoint': 'data/models/rat_genecompass_init_v2', 'vocab_size': 55275, 'num_species': 3, 'hidden_size': 768, 'num_hidden_layers': 12, 'num_attention_heads': 12, 'intermediate_size': 3072, 'max_position_embeddings': 2048}
training: {'weight_decay': 0.01, 'fp16': True, 'gradient_checkpointing': True, 'emb_warmup_steps': 10000}
mixing: {'rat_prob': 0.4, 'human_prob': 0.45, 'mouse_prob': 0.15, 'seed': 42, 'exclude_studies': []}
logging: {'logging_steps': 50, 'save_strategy': 'steps', 'save_steps': 10000, 'save_total_limit': 5, 'wandb_project': 'motrpac-genecompass-rat'}
distributed: {'nproc_per_node': 4, 'nnodes': 1}


In [15]:
import os, json

FINAL_MODEL = ('../data/models/'
               'rat_genecompass_finetuned_v2/models/rat_phase2_mixed_species/models')

model_bin = f'{FINAL_MODEL}/pytorch_model.bin'

print("Final model verification")
print(f"  pytorch_model.bin: {os.path.getsize(model_bin)/1e9:.2f} GB")
print(f"  Exists:            {os.path.exists(model_bin)}")
print()

# Phase 2 checkpoint summary
p2_dir = f'{FINAL_MODEL}/..'
ckpts = sorted([d for d in os.listdir(p2_dir) if d.startswith('checkpoint-')], key=lambda d: int(d.split('-')[1]))
print(f"Phase 2 checkpoints ({len(ckpts)}):")
for ck in ckpts:
    size_b = os.path.getsize(f'{p2_dir}/{ck}/pytorch_model.bin')
    print(f"  {ck:<30}  {size_b/1e9:.2f} GB  {'← final' if ck == ckpts[-1] else ''}")

Final model verification
  pytorch_model.bin: 1.22 GB
  Exists:            True

Phase 2 checkpoints (5):
  checkpoint-560000               1.22 GB  
  checkpoint-570000               1.22 GB  
  checkpoint-580000               1.22 GB  
  checkpoint-590000               1.22 GB  
  checkpoint-591763               1.22 GB  ← final
